# Prompt 1 completion — resumable 480-case stages

This notebook imports legacy OOF provenance without an exact-reproduction claim, completes only missing SDFs, and resumes audit/identity per case. It does not retrain Track A, modify checkpoints, require true softmax, or open the 52-case S cohort.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys

# Single editable configuration cell.
DRIVE_ROOT = Path('/content/drive/MyDrive/ToothFairy/ToothFairy3/iac_runs')
DATASET_ROOT = DRIVE_ROOT / 'dataset_cache_colab_v1/Dataset801_IAC_LR'
NNUNET_RESULTS = DRIVE_ROOT / 'nnUNet_results'
TRACKB_CACHE_ROOT = DRIVE_ROOT / 'sdf_cache_backup'
OUTPUT_ROOT = DRIVE_ROOT / 'outputs'
SPLITS_PATH = DRIVE_ROOT / 'configs_cache/splits.json'
REPO_URL = 'https://github.com/ColdVI/ToothFairy3-IAC-Segmentation-Flow.git'
PINNED_COMMIT = 'REPLACE_WITH_PROMPT1_COMMIT_SHA'
NUM_WORKERS = 2
DEVICE = 'cuda'
QUICK_PREFLIGHT_CASES_PER_FOLD = 1
FULL_PREFLIGHT_CASES = 40
PREFLIGHT_MODE = 'quick'
MAX_CASES = 2  # retained for config compatibility; smoke is explicitly two cases
SDF_BATCH_SIZE = 12  # must be in [8, 16]; batches are staged on /content SSD
FORCE_REBUILD = False
EXPORT_TRUE_SOFTMAX = False  # optional; not required for the identity baseline
RETRY_FAILED = True

REPO = Path('/content/ToothFairy3-IAC-Segmentation-Flow')
CONFIG_PATH = Path('/content/prompt1_completion_config.json')

def bootstrap_repo():
    if PINNED_COMMIT.startswith('REPLACE_'):
        raise ValueError('Set PINNED_COMMIT to the delivered Prompt-1 commit SHA')
    if not REPO.is_dir():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
    subprocess.run(['git', 'fetch', '--all', '--tags'], cwd=REPO, check=True)
    subprocess.run(['git', 'checkout', '--detach', PINNED_COMMIT], cwd=REPO, check=True)
    head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
    wanted = subprocess.check_output(['git', 'rev-parse', PINNED_COMMIT], cwd=REPO, text=True).strip()
    assert head == wanted, (head, wanted)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO/'requirements.txt')], check=True)
    return REPO

def write_runner_config():
    values = {
        'DRIVE_ROOT': DRIVE_ROOT, 'DATASET_ROOT': DATASET_ROOT,
        'NNUNET_RESULTS': NNUNET_RESULTS, 'TRACKB_CACHE_ROOT': TRACKB_CACHE_ROOT,
        'OUTPUT_ROOT': OUTPUT_ROOT, 'SPLITS_PATH': SPLITS_PATH,
        'PINNED_COMMIT': PINNED_COMMIT, 'NUM_WORKERS': NUM_WORKERS,
        'DEVICE': DEVICE, 'QUICK_PREFLIGHT_CASES_PER_FOLD': QUICK_PREFLIGHT_CASES_PER_FOLD,
        'FULL_PREFLIGHT_CASES': FULL_PREFLIGHT_CASES, 'PREFLIGHT_MODE': PREFLIGHT_MODE,
        'MAX_CASES': MAX_CASES, 'SDF_BATCH_SIZE': SDF_BATCH_SIZE,
        'FORCE_REBUILD': FORCE_REBUILD,
        'EXPORT_TRUE_SOFTMAX': EXPORT_TRUE_SOFTMAX, 'RETRY_FAILED': RETRY_FAILED}
    payload = {name: str(value) if isinstance(value, Path) else value for name, value in values.items()}
    CONFIG_PATH.write_text(json.dumps(payload, indent=2))
    return CONFIG_PATH


## A. Setup + pytest

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
REPO = bootstrap_repo()
import torch
assert DEVICE == 'cuda' and torch.cuda.is_available(), 'A CUDA runtime is required for OOF inference'
for name, path in [('DATASET_ROOT', DATASET_ROOT), ('NNUNET_RESULTS', NNUNET_RESULTS), ('SPLITS_PATH', SPLITS_PATH)]:
    assert path.exists(), f'{name} missing: {path}'
for path in (TRACKB_CACHE_ROOT, OUTPUT_ROOT):
    path.mkdir(parents=True, exist_ok=True)
    assert path.resolve().is_relative_to(DRIVE_ROOT.resolve())
cfg = write_runner_config()
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests'], cwd=REPO, check=True)


## B. Build inventory
Stats all 480 cases, but reuses cached metadata/checksums and does not reopen unchanged NIfTI/NPZ files.

In [ ]:
cfg = write_runner_config()
subprocess.run([sys.executable, 'scripts/prompt1_completion.py', '--config', str(cfg), 'build-inventory'], cwd=REPO, check=True)


## C. Import legacy provenance
Records expected folds, frozen checkpoint hashes, and the L4-vs-A100 near-exact diagnostic. Legacy hard artifacts remain untouched.

In [ ]:
cfg = write_runner_config()
subprocess.run([sys.executable, 'scripts/prompt1_completion.py', '--config', str(cfg), 'import-legacy-provenance'], cwd=REPO, check=True)


## D. Audit 480
Checks split leakage, provenance, geometry, validity, and hard-to-SDF round-trip. Missing true softmax is informational only.

In [ ]:
cfg = write_runner_config()
subprocess.run([sys.executable, 'scripts/prompt1_completion.py', '--config', str(cfg), 'audit-480'], cwd=REPO, check=True)


## E. Complete missing SDF
Stages 8–16 cases at a time on local SSD, computes only missing/invalid caches, validates locally, and atomically publishes each completed case.

In [ ]:
cfg = write_runner_config()
subprocess.run([sys.executable, 'scripts/prompt1_completion.py', '--config', str(cfg), 'complete-missing-sdf'], cwd=REPO, check=True)


## F. Re-audit completed cache
Refreshes inventory and re-evaluates only cases whose input signatures changed.

In [ ]:
cfg = write_runner_config()
subprocess.run([sys.executable, 'scripts/prompt1_completion.py', '--config', str(cfg), 'build-inventory'], cwd=REPO, check=True)
subprocess.run([sys.executable, 'scripts/prompt1_completion.py', '--config', str(cfg), 'audit-480'], cwd=REPO, check=True)


## G. Identity 480
Runs direct hard, SDF sign, and zero-velocity full-path evaluation with case-level resume state.

In [ ]:
cfg = write_runner_config()
subprocess.run([sys.executable, 'scripts/prompt1_completion.py', '--config', str(cfg), 'identity-480'], cwd=REPO, check=True)


## H. Finalize Prompt 1
Verifies stage receipts/checksums and sets complete_cv=true only for a passing 480-case result. The non-inferiority margin remains null.

In [ ]:
cfg = write_runner_config()
subprocess.run([sys.executable, 'scripts/prompt1_completion.py', '--config', str(cfg), 'finalize-prompt1'], cwd=REPO, check=True)


## I. Full wrapper (optional)
Runs the same resumable stages in order. Re-running it skips unchanged inventory, audited cases, SDFs, and identity cases.

In [ ]:
cfg = write_runner_config()
subprocess.run([sys.executable, 'scripts/prompt1_completion.py', '--config', str(cfg), 'run-full'], cwd=REPO, check=True)
# Outputs include prompt1/cache_inventory_480.json and baselines/identity_prior.json.
